In [4]:
import holidays
import numpy
import numpy as np
import pandas
import config

UK_HOLIDAYS = pandas.to_datetime(list(holidays.country_holidays('UK', years=range(2011, 2026)).keys()))

raw_feature_name, target_feature_name = zip(*[
    ('Date', 'date'),
    ('Time', 'time'),
    ('Ozone', 'O3'),
    ('Nitric oxide', 'NO'),
    ('Nitrogen dioxide', 'NO2'),
    ('Carbon monoxide', 'CO'),
    ('Modelled Wind Direction', 'wind_dir'),
    ('Modelled Wind Speed', 'wind_speed'),
    ('Modelled Temperature', 'temp'),
    ('PM10 particulate matter (Hourly measured)', 'PM10'),
    ('PM2.5 particulate matter (Hourly measured)', 'PM2.5')
])


def add_day_category(df, country):
    current_date = df['date']
    next_date = current_date + pandas.Timedelta(days=1)
    is_off_day = lambda date: is_weekend(date) | is_holiday(date, country)

    category = pandas.Categorical(numpy.select(
        [is_off_day(current_date), is_off_day(next_date)],
        [1, 2],
        default=0
    ))

    df.insert(df.columns.get_loc('date') + 1, 'day_category', category)
    return df


def is_weekend(date):
    return date.dt.day_name().isin(['Saturday', 'Sunday'])


def is_holiday(date, country_holidays):
    return date.isin(country_holidays)


def apply_min_max(df, exclude=None):
    x = df.select_dtypes(include='number')
    if exclude:
        x = x.drop(columns=exclude, errors='ignore')
    df[x.columns] = (2 * (x - x.min()) / (x.max() - x.min()) - 1).round(4)
    return df


def process(csv_year):
    print(f"Processing {csv_year}")
    return (
        pandas.read_csv(
            f"{config.raw_csv}/{csv_year}.csv",
            na_values=['No data'],
            parse_dates=['Date'],
            skiprows=config.rows_to_skip,
            skipfooter=1,
            usecols=raw_feature_name,
            engine='python'
        )
        .rename(columns=dict(zip(raw_feature_name, target_feature_name)))
        .assign(time=lambda df: df['time'].str[:2].astype(int))
        .pipe(add_day_category, UK_HOLIDAYS)
        .assign(wind_dir_sin=lambda df: np.sin(np.radians(df['wind_dir'])).round(4),
                wind_dir_cos=lambda df: np.cos(np.radians(df['wind_dir'])).round(4))
        .drop(columns=['wind_dir'])
    )

In [5]:
combined_raw = (
    pandas.concat([process(year) for year in range(config.start_year, config.end_year + 1)])
    .sort_values(['date', 'time'])
)
combined_raw.to_csv(config.unscaled_csv, index=False)

Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025


In [6]:
combined_scaled = (combined_raw
                   .drop(columns=['CO'])
                   .pipe(apply_min_max, exclude=['wind_dir_sin', 'wind_dir_cos', 'day_category', 'time'])
                   .dropna())

combined_scaled.to_csv(config.scaled_csv, index=False)

In [2]:
features = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir_sin', 'wind_dir_cos']

def make_blocks(df, start_hour, n_hours, day_category):
    target_hours = list(range(start_hour, start_hour + n_hours))

    filtered = df[
        (df['day_category'] == day_category) &
        (df['time'].isin(target_hours))
    ]

    blocks = []
    dates = []

    for date, group in filtered.groupby('date'):
        if len(group) != n_hours:
            continue

        row = group.sort_values('time')[features].values.flatten()
        blocks.append(row)
        dates.append(date)

    return np.array(blocks), dates

In [3]:
blocks, dates = make_blocks(combined_scaled, start_hour=9, n_hours=5, day_category=0)
print(f"Blokų: {len(blocks)}")
print(f"Feature vektoriaus ilgis: {blocks.shape[1]}")  # 5 * 9 = 45
print(f"Feature vektoriaus ilgis: {blocks[1]}")  # 5 * 9 = 45

Blokų: 1950
Feature vektoriaus ilgis: 45
Feature vektoriaus ilgis: [-9.373e-01 -8.300e-02 -2.460e-02 -5.608e-01 -5.023e-01 -3.723e-01
 -1.253e-01 -9.977e-01  6.800e-02 -9.373e-01 -1.471e-01  3.000e-04
 -6.237e-01 -5.023e-01 -3.723e-01 -1.071e-01 -9.851e-01  1.719e-01
 -9.373e-01 -2.135e-01 -4.940e-02 -6.342e-01 -4.570e-01 -4.161e-01
 -7.520e-02 -9.636e-01  2.672e-01 -9.373e-01 -4.174e-01 -1.675e-01
 -6.447e-01 -5.173e-01 -4.453e-01 -7.060e-02 -9.603e-01  2.790e-01
 -9.373e-01 -3.189e-01 -1.302e-01 -5.503e-01 -6.078e-01 -5.766e-01
 -7.060e-02 -9.912e-01  1.323e-01]
